In [1]:
# 1138-01  Use 2024 computer rates

In [1]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
import numpy as np
import pandas as pd
import glob
import re
import matplotlib.pyplot as plt
import os
import pyvista as pv
from pathlib import Path

pd.set_option('display.max_columns', None)


In [2]:
spath = r"C:\Users\cakyol\gpkgs\**.gpkg"
files = [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

#spath = r"C:\scipts\1138-01\Sonar\PPG2\1993\cynthia\*.gpkg"
#files = files + [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

print(len(files))
df = pd.DataFrame({'path':files})
df['fname'] = df['path'].apply(lambda x: x.split('/')[-1])
df['orient'] = 'v'
df['orient'] = df['orient'].where(df.fname.apply( lambda x:'ft' not in x), other='h')

# Drop Bad row
df.drop(index=[14], inplace=True)

## well
import re

df['Well_No'] = None

for index, row in df.iterrows():

    fname = row['fname']
    fname_upper = fname.upper()

    # -------------------------
    # HORIZONTAL FILES
    # -------------------------
    if row['orient'] == 'h':

        m = re.search(r'\d{4}', fname)
        if m:
            df.at[index, 'z'] = int(m.group())

        df.at[index, 'Well_No'] = 'all'


    # -------------------------
    # VERTICAL FILES
    # -------------------------
    else:

        # extract azimuths
        m = re.search(r'\d+-\d+', fname)
        if m:
            a0, a1 = m.group().split('-')
            df.at[index, 'azi0'] = int(a0)
            df.at[index, 'azi1'] = int(a1)

        # assign well based on filename
        if 'ALL' in fname_upper:
            df.at[index, 'Well_No'] = 'all'

        elif 'PPG4' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG4'

        elif 'PPG2' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG2'

        else:
            df.at[index, 'Well_No'] = 'UNKNOWN'

25


In [3]:
df

,path,fname,orient,Well_No,azi0,azi1,z
0,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,0-180_PPG2.gpkg,v,PPG2,0.0,180.0,NaN
1,C:/Users/cakyol/gpkgs/0-180_PPG4.gpkg,0-180_PPG4.gpkg,v,PPG4,0.0,180.0,NaN
2,C:/Users/cakyol/gpkgs/225-45_PPG2.gpkg,225-45_PPG2.gpkg,v,PPG2,225.0,45.0,NaN
3,C:/Users/cakyol/gpkgs/225-45_PPG4.gpkg,225-45_PPG4.gpkg,v,PPG4,225.0,45.0,NaN
4,C:/Users/cakyol/gpkgs/2600ft.gpkg,2600ft.gpkg,h,all,NaN,NaN,2600.0
5,C:/Users/cakyol/gpkgs/270-90_PPG2.gpkg,270-90_PPG2.gpkg,v,PPG2,270.0,90.0,NaN
6,C:/Users/cakyol/gpkgs/270-90_PPG4_1.gpkg,270-90_PPG4_1.gpkg,v,PPG4,270.0,90.0,NaN
7,C:/Users/cakyol/gpkgs/270-90_PPG4_2.gpkg,270-90_PPG4_2.gpkg,v,PPG4,270.0,90.0,NaN
8,C:/Users/cakyol/gpkgs/2700ft.gpkg,2700ft.gpkg,h,all,NaN,NaN,2700.0
9,C:/Users/cakyol/gpkgs/277-97_PPG2.gpkg,277-97_PPG2.gpkg,v,PPG2,277.0,97.0,NaN


In [4]:
## dX, dY
# ppg 2 
2624245.599498,643042.320207

# ppg 4
2624685.606284,642633.314476

#horizontal offsets
#ppg2 dy:155, dx:-290
#ppg4 dy:-120, dx:180

(2624685.606284, 642633.314476)

In [5]:
# =========================
# USER INPUTS
# =========================
CRS_OUT = "EPSG:3452"

# Main 0-reference point for the cavern
CAVERN_CENTER = {
    "X0": 2624465.602891,
    "Y0": 642837.817341
}

# Wellhead offsets from cavern center
# dx = East-West difference
# dy = North-South difference
WELL_OFFSETS_FROM_CAVERN_CENTER = {
    "PPG2": {
        "dx": -300,
        "dy":  155
    },
    "PPG4": {
        "dx":  180,
        "dy": -120
    }
}

# Optional: rebuild WELLHEADS automatically from center + offsets
WELLHEADS = {
    well: {
        "Xwh": CAVERN_CENTER["X0"] + off["dx"],
        "Ywh": CAVERN_CENTER["Y0"] + off["dy"]
    }
    for well, off in WELL_OFFSETS_FROM_CAVERN_CENTER.items()
}

SCALE_LOCAL_TO_MAP = 1.0
STEP = 10.0
VERT_AZ_MODE = "mean"
VERTICAL_X_IS_YLOCAL = True

OUT_GPKG = "PPG_1993_referenced_pointclouds2.gpkg"
OUT_LAYER = "sonar_points"

In [6]:
# =========================
# Helper functions
# =========================
def circular_mean_deg(a, b):
    a = np.deg2rad(a); b = np.deg2rad(b)
    x = np.cos(a) + np.cos(b)
    y = np.sin(a) + np.sin(b)
    return (np.rad2deg(np.arctan2(y, x)) + 360) % 360

def get_vertical_azimuths(row):
    a0, a1 = row.get("azi0", np.nan), row.get("azi1", np.nan)
    if pd.isna(a0) and pd.isna(a1):
        return []
    if VERT_AZ_MODE == "azi0":
        return [float(a0)]
    if VERT_AZ_MODE == "azi1":
        return [float(a1)]
    if VERT_AZ_MODE == "mean":
        return [float(circular_mean_deg(float(a0), float(a1)))]
    if VERT_AZ_MODE == "both":
        out = []
        if not pd.isna(a0): out.append(float(a0))
        if not pd.isna(a1): out.append(float(a1))
        return out
    raise ValueError("VERT_AZ_MODE must be one of: azi0, azi1, mean, both")

def sample_geom_xy(geom, step=STEP):
    """Sample boundary points from Polygon/LineString into Nx2 array."""
    if geom is None or geom.is_empty:
        return np.empty((0, 2), dtype=float)

    if isinstance(geom, Polygon):
        line = geom.exterior
    elif isinstance(geom, LineString):
        line = geom
    else:
        b = geom.boundary
        if isinstance(b, LineString):
            line = b
        else:
            return np.empty((0, 2), dtype=float)

    L = line.length
    if L <= 0:
        return np.empty((0, 2), dtype=float)

    n = max(2, int(np.ceil(L / step)))
    dists = np.linspace(0, L, n)
    pts = [line.interpolate(d) for d in dists]
    return np.array([[p.x, p.y] for p in pts], dtype=float)

def cavern_center_from_wells():
    centers = {
        well: (
            CAVERN_CENTER["X0"],
            CAVERN_CENTER["Y0"]
        )
        for well in WELL_OFFSETS_FROM_CAVERN_CENTER.keys()
    }
    return (CAVERN_CENTER["X0"], CAVERN_CENTER["Y0"]), centers

def wellhead_from_center(well):
    Xw = CAVERN_CENTER["X0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dx"]
    Yw = CAVERN_CENTER["Y0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dy"]
    return Xw, Yw


In [7]:
def section_origin_from_well(well, az_deg, offset):
    Xw, Yw = wellhead_from_center(well)
    az = np.deg2rad(float(az_deg))
    X0 = Xw - offset * np.sin(az)
    Y0 = Yw - offset * np.cos(az)
    return X0, Y0

Xsec_from_ppg2, Ysec_from_ppg2 = section_origin_from_well("PPG2", 300, -330)
Xsec_from_ppg4, Ysec_from_ppg4 = section_origin_from_well("PPG4", 300, 210)

print("300-120 origin from PPG2:", Xsec_from_ppg2, Ysec_from_ppg2)
print("300-120 origin from PPG4:", Xsec_from_ppg4, Ysec_from_ppg4)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg4 - Ysec_from_ppg4)

Xsec_300120 = 0.5 * (Xsec_from_ppg2 + Xsec_from_ppg4)
Ysec_300120 = 0.5 * (Ysec_from_ppg2 + Ysec_from_ppg4)

print("Using 300-120 section origin:", Xsec_300120, Ysec_300120)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg2 - Ysec_from_ppg4)

300-120 origin from PPG2: 2623879.8145077513 643157.817341
300-120 origin from PPG4: 2624827.468225795 642612.817341
Difference: -947.6537180435844 0.0
Using 300-120 section origin: 2624353.6413667733 642885.317341
Difference: -947.6537180435844 545.0


In [8]:
# Check cavern center computed from each well
for w in ["PPG2", "PPG4"]:
    Xw, Yw = wellhead_from_center(w)
    print(w, "wellhead:", Xw, Yw)

print("Cavern center:", CAVERN_CENTER["X0"], CAVERN_CENTER["Y0"])

Xc2 = WELLHEADS["PPG2"]["Xwh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG2"]["dx"]
Yc2 = WELLHEADS["PPG2"]["Ywh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG2"]["dy"]
Xc4 = WELLHEADS["PPG4"]["Xwh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG4"]["dx"]
Yc4 = WELLHEADS["PPG4"]["Ywh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG4"]["dy"]

print("Center diff PPG2-PPG4:", (Xc2-Xc4), (Yc2-Yc4))

PPG2 wellhead: 2624165.602891 642992.817341
PPG4 wellhead: 2624645.602891 642717.817341
Cavern center: 2624465.602891 642837.817341
Center diff PPG2-PPG4: 0.0 0.0


In [9]:
# =========================
# Build referenced point cloud + metadata
# =========================


(Xcav, Ycav), centers = cavern_center_from_wells()
print("Cavern center from wells (per-well):", centers)
print("Using cavern center (average):", (Xcav, Ycav))


Xsec = 0.5 * (WELLHEADS["PPG2"]["Xwh"] + WELLHEADS["PPG4"]["Xwh"])
Ysec = 0.5 * (WELLHEADS["PPG2"]["Ywh"] + WELLHEADS["PPG4"]["Ywh"])

print("Section midpoint for ALL verticals:", (Xsec, Ysec))

# ---------------------------------
# Fit cavern center for 300-120_all
# ---------------------------------
az_300120 = np.deg2rad(120.0)

# distances along section axis from cavern center
s_ppg2 = -330.0
s_ppg4 = 290.0

Xcav2 = WELLHEADS["PPG2"]["Xwh"] - s_ppg2 * np.sin(az_300120)
Ycav2 = WELLHEADS["PPG2"]["Ywh"] - s_ppg2 * np.cos(az_300120)

Xcav4 = WELLHEADS["PPG4"]["Xwh"] - s_ppg4 * np.sin(az_300120)
Ycav4 = WELLHEADS["PPG4"]["Ywh"] - s_ppg4 * np.cos(az_300120)

Xcav_fit = 0.5 * (Xcav2 + Xcav4)
Ycav_fit = 0.5 * (Ycav2 + Ycav4)

print("Fitted cavern center for 300-120:", (Xcav_fit, Ycav_fit))

records = []

for _, row in df.iterrows():

    orient = row["orient"]
    fname  = row["fname"]
    path   = row["path"]
    wellno = str(row.get("Well_No", "all")).lower()

    gdf = gpd.read_file(path)
    if gdf.empty:
        continue

    # -------------------------
    # HORIZONTAL: cavern-centered
    # -------------------------
    if orient == "h":

        depth = row.get("z", np.nan)
        if pd.isna(depth):
            continue

        for geom in gdf.geometry:
            xy = sample_geom_xy(geom, step=STEP)
            if xy.size == 0:
                continue

            x_local = xy[:, 0]
            y_local = xy[:, 1]

            X = Xcav + x_local
            Y = Ycav + y_local
            Z = np.full_like(X, -float(depth), dtype=float)

            for xi, yi, zi in zip(X, Y, Z):
                records.append({
                    "fname": fname,
                    "path": path,
                    "orient": "h",
                    "depth": float(depth),
                    "azi0": np.nan,
                    "azi1": np.nan,
                    "az_used": np.nan,
                    "Well_No": wellno,
                    "geometry": Point(float(xi), float(yi), float(zi)),
                })

    # -------------------------
    # VERTICAL
    # -------------------------
    elif orient == "v":

        az_list = get_vertical_azimuths(row)
        if not az_list:
            continue

        a0 = row.get("azi0", np.nan)
        a1 = row.get("azi1", np.nan)

        # choose anchor
        if wellno == "ppg2":
            X0 = WELLHEADS["PPG2"]["Xwh"]
            Y0 = WELLHEADS["PPG2"]["Ywh"]

        elif wellno == "ppg4":
            X0 = WELLHEADS["PPG4"]["Xwh"]
            Y0 = WELLHEADS["PPG4"]["Ywh"]

        elif wellno == "all":
            if "300-120" in fname:
                X0 = Xcav_fit
                Y0 = Ycav_fit
            else:
                X0 = Xsec
                Y0 = Ysec

        else:
            print(f"Skipping {fname}: unknown Well_No = {wellno}")
            continue

        for az_deg in az_list:

#ROTATING WITH RELATIVE AZIMUTH************************************************
#300-120_all
            if "300-120" in fname:
                az_use = 120.0
#277-97_PPG2
            elif "277-97" in fname and wellno == "ppg2":
                # rotate relative to original azimuth, keep wellhead fixed
                az_use = (float(az_deg) + 90.0) % 360
                print("ROTATING 277-97_PPG2:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_2
            elif fname == "270-90_PPG4_2.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_1
            elif fname == "270-90_PPG4_1.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG4
            elif fname == "315-135_PPG4.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG2
            elif fname == "315-135_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
            else:
                az_use = float(az_deg)

            az = np.deg2rad(az_use)
#*****************************************************************************
#SHIFTS

    
            x_shift = 0.0
            y_shift = 0.0

            for geom in gdf.geometry:
                xy = sample_geom_xy(geom, step=STEP)
                if xy.size == 0:
                    continue

                if VERTICAL_X_IS_YLOCAL:
                    y_local = xy[:, 0]
                    z_local = xy[:, 1]
                else:
                    y_local = xy[:, 1]
                    z_local = xy[:, 0]

                dX = y_local * np.sin(az)
                dY = y_local * np.cos(az)

                X = X0 + dX + x_shift
                Y = Y0 + dY + y_shift
                Z = z_local.astype(float)

                for xi, yi, zi in zip(X, Y, Z):
                    records.append({
                        "fname": fname,
                        "path": path,
                        "orient": "v",
                        "depth": np.nan,
                        "azi0": float(a0) if not pd.isna(a0) else np.nan,
                        "azi1": float(a1) if not pd.isna(a1) else np.nan,
                        "az_used": az_use,
                        "Well_No": wellno,
                        "geometry": Point(float(xi), float(yi), float(zi)),
                    })

gdf_out = gpd.GeoDataFrame(records, geometry="geometry", crs=CRS_OUT)
print("Total referenced points:", len(gdf_out))

Cavern center from wells (per-well): {'PPG2': (2624465.602891, 642837.817341), 'PPG4': (2624465.602891, 642837.817341)}
Using cavern center (average): (2624465.602891, 642837.817341)
Section midpoint for ALL verticals: (2624405.602891, 642855.317341)
Fitted cavern center for 300-120: (2624422.923399076, 642845.317341)
ROTATING: 270-90_PPG4_1.gpkg old: 180.0 new: 90.0
ROTATING: 270-90_PPG4_2.gpkg old: 180.0 new: 90.0
ROTATING 277-97_PPG2: 277-97_PPG2.gpkg old: 19.17900802581073 new: 109.17900802581073
ROTATING: 315-135_PPG2.gpkg old: 225.0 new: 135.0
ROTATING: 315-135_PPG4.gpkg old: 225.0 new: 135.0
Total referenced points: 6548


In [10]:

def shift_point_3d(p, dx=0.0, dy=0.0, dz=0.0):
    return Point(p.x + dx, p.y + dy, p.z + dz)

def rotate_point_xy_around_anchor(p, angle_deg, x0, y0):
    """
    Rotate a 3D point in the XY plane around anchor (x0, y0).
    Z stays unchanged.
    Positive angle = counterclockwise.
    """
    ang = np.deg2rad(angle_deg)

    dx = p.x - x0
    dy = p.y - y0

    xr = dx * np.cos(ang) - dy * np.sin(ang)
    yr = dx * np.sin(ang) + dy * np.cos(ang)

    return Point(x0 + xr, y0 + yr, p.z)

def transform_gdf_points(
    gdf,
    mask,
    angle_deg=0.0,
    anchor_x=None,
    anchor_y=None,
    dx=0.0,
    dy=0.0,
    dz=0.0
):
    gdf2 = gdf.copy()

    def _transform(p):
        p2 = p

        if angle_deg != 0.0:
            if anchor_x is None or anchor_y is None:
                raise ValueError("anchor_x and anchor_y are required for rotation.")
            p2 = rotate_point_xy_around_anchor(p2, angle_deg, anchor_x, anchor_y)

        if dx != 0.0 or dy != 0.0 or dz != 0.0:
            p2 = shift_point_3d(p2, dx=dx, dy=dy, dz=dz)

        return p2

    gdf2.loc[mask, "geometry"] = gdf2.loc[mask, "geometry"].apply(_transform)
    return gdf2

In [11]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt

plotter = pv.Plotter()

files = gdf_out["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_out[gdf_out["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    # Highlight important sections
    if "315-135_PPG4" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )

    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26b39d1c190_0&reconnect=auto" class="pyvis…

In [12]:
# -------------------------------
# EDIT ONE VERTICAL SECTION AFTER BUILD
# -------------------------------

#Bullk shift of 300-120_all.gpkg

mask = (
    (gdf_out["orient"] == "v") &
    (gdf_out["fname"].str.contains("300-120", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_out,
    mask=mask,
    angle_deg=0.0,         # try 2, -2, 5, etc.
    anchor_x=Xcav_fit,     # or Xsec / PPG2 / PPG4 wellhead
    anchor_y=Ycav_fit,
    dx=30.0,
    dy=-28.0,
    dz=0.0
)

#Rotating 315-135_PPG4 (includes 1981 sonar)
# second fix
mask = (
    (gdf_edit["orient"] == "v") &
    (gdf_edit["fname"].str.contains("277-97_PPG2", case=False, na=False)) &
    (gdf_edit["Well_No"] == "ppg2")
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,
    anchor_x=WELLHEADS["PPG2"]["Xwh"],
    anchor_y=WELLHEADS["PPG2"]["Ywh"],
    dx=280.0,
    dy=-125.0,
    dz=0.0
)
# -------------------------------
# SHIFT 2700ft HORIZONTAL TO 2900ft
# -------------------------------

mask = (
    (gdf_edit["orient"] == "h") &
    (gdf_edit["fname"].str.contains("2700", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,   # no rotation
    anchor_x=0,
    anchor_y=0,
    dx=0.0,
    dy=0.0,
    dz=-50.0        # move from 2700 to 2900
)


In [13]:


plotter = pv.Plotter()

files = gdf_edit["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_edit[gdf_edit["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    if "315-135_PPG2" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )
    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26b3d89e510_1&reconnect=auto" class="pyvis…

In [ ]:
#CREATING SYNTHETIC DATA


In [14]:

from shapely.ops import polygonize, unary_union
from scipy.spatial import Delaunay



# =========================================================
# HELPERS
# =========================================================

def extract_xyz_from_gdf(gdf):
    out = gdf.copy()
    out["x"] = out.geometry.x
    out["y"] = out.geometry.y
    out["z"] = out.geometry.apply(lambda g: g.z if g.has_z else np.nan)
    out = out.dropna(subset=["x", "y", "z"]).copy()
    return out


def boundary_from_delaunay(points_xy, max_edge_length=None):
    """
    Build outer boundary from 2D Delaunay triangles.
    Keeps only triangles whose 3 edges are <= max_edge_length.
    Then extracts boundary edges (edges appearing only once).
    """
    if len(points_xy) < 3:
        return None

    tri = Delaunay(points_xy)
    simplices = tri.simplices

    edge_count = {}
    kept_any = False

    for simp in simplices:
        p0, p1, p2 = points_xy[simp]

        e01 = np.linalg.norm(p0 - p1)
        e12 = np.linalg.norm(p1 - p2)
        e20 = np.linalg.norm(p2 - p0)

        if max_edge_length is not None:
            if max(e01, e12, e20) > max_edge_length:
                continue

        kept_any = True
        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0]))),
        ]

        for e in edges:
            edge_count[e] = edge_count.get(e, 0) + 1

    if not kept_any:
        return None

    boundary_edges = [e for e, c in edge_count.items() if c == 1]
    if not boundary_edges:
        return None

    lines = [LineString([points_xy[i], points_xy[j]]) for i, j in boundary_edges]
    merged = unary_union(lines)
    polys = list(polygonize(merged))

    if not polys:
        return None

    # Take largest polygon
    poly = max(polys, key=lambda p: p.area)
    return poly


def resample_closed_ring(xy, n_samples=180):
    """
    Resample a closed XY boundary to fixed point count.
    """
    if len(xy) < 4:
        return None

    # ensure closed
    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seg = np.sqrt(np.sum(np.diff(xy, axis=0) ** 2, axis=1))
    s = np.concatenate([[0], np.cumsum(seg)])
    total = s[-1]

    if total == 0:
        return None

    target = np.linspace(0, total, n_samples + 1)[:-1]
    x_new = np.interp(target, s, xy[:, 0])
    y_new = np.interp(target, s, xy[:, 1])

    return np.column_stack([x_new, y_new])


def rotate_ring_to_reference(ring_xy, ref_xy):
    """
    Circularly shift ring_xy so it best matches ref_xy.
    Both must have same shape (N,2).
    """
    n = len(ring_xy)
    best_shift = 0
    best_score = np.inf

    for k in range(n):
        test = np.roll(ring_xy, shift=k, axis=0)
        score = np.mean(np.sum((test - ref_xy) ** 2, axis=1))
        if score < best_score:
            best_score = score
            best_shift = k

    return np.roll(ring_xy, shift=best_shift, axis=0)


def ensure_same_direction(ring_xy, ref_xy):
    """
    Flip ring if reversed ordering matches ref better.
    """
    forward = np.mean(np.sum((ring_xy - ref_xy) ** 2, axis=1))
    reverse = np.mean(np.sum((ring_xy[::-1] - ref_xy) ** 2, axis=1))
    return ring_xy if forward <= reverse else ring_xy[::-1]



In [15]:
# =========================================================
# MAIN: BUILD SYNTHETIC SLICES USING DELAUNAY
# =========================================================

def build_delaunay_slices(
    gdf,
    z_step=2.5,              # output slice spacing
    dz_window=2.5,           # half-thickness of each slice band
    n_boundary_pts=180,      # resampled points per slice
    min_points_per_slice=12,
    max_edge_length=None     # set to something like 60 or 80 if needed
):
    """
    Slice full point cloud by z, build Delaunay boundary per slice,
    resample to a fixed number of points, and align neighboring rings.
    """
    work = extract_xyz_from_gdf(gdf)

    zmin = work["z"].min()
    zmax = work["z"].max()
    z_levels = np.arange(zmin, zmax + z_step, z_step)

    rings = []
    ref_ring = None

    for z0 in z_levels:
        sl = work[(work["z"] >= z0 - dz_window) & (work["z"] <= z0 + dz_window)].copy()

        if len(sl) < min_points_per_slice:
            continue

        pts = sl[["x", "y"]].to_numpy()

        poly = boundary_from_delaunay(pts, max_edge_length=max_edge_length)
        if poly is None or poly.area <= 0:
            continue

        xy = np.array(poly.exterior.coords)
        ring = resample_closed_ring(xy, n_samples=n_boundary_pts)
        if ring is None:
            continue

        # Align with previous ring so mesh does not twist
        if ref_ring is not None:
            ring = ensure_same_direction(ring, ref_ring)
            ring = rotate_ring_to_reference(ring, ref_ring)

        ref_ring = ring.copy()

        ring_df = pd.DataFrame({
            "x": ring[:, 0],
            "y": ring[:, 1],
            "z": z0,
            "z_slice": z0,
            "source": "synthetic_delaunay_slice",
            "slice_n_input_pts": len(sl)
        })
        rings.append(ring_df)

    if not rings:
        print("No valid slices were created.")
        return None, None

    syn_df = pd.concat(rings, ignore_index=True)
    syn_df["geometry"] = [Point(x, y, z) for x, y, z in zip(syn_df["x"], syn_df["y"], syn_df["z"])]
    syn_gdf = gpd.GeoDataFrame(syn_df, geometry="geometry", crs=gdf.crs)

    return syn_gdf, z_levels



In [16]:
# =========================================================
# BUILD TRIANGULATED SURFACE MESH BETWEEN SLICES
# =========================================================

def mesh_between_slices(syn_gdf, n_boundary_pts):
    """
    Build PyVista surface mesh by connecting adjacent z-rings.
    Assumes each slice has exactly n_boundary_pts points.
    """
    syn = syn_gdf.copy().sort_values(["z_slice"]).reset_index(drop=True)
    z_vals = np.sort(syn["z_slice"].unique())

    ring_arrays = []
    for z0 in z_vals:
        sub = syn[syn["z_slice"] == z0].copy()
        if len(sub) != n_boundary_pts:
            continue
        ring_xyz = sub[["x", "y", "z"]].to_numpy()
        ring_arrays.append((z0, ring_xyz))

    if len(ring_arrays) < 2:
        print("Need at least 2 valid slices to mesh.")
        return None

    points = []
    faces = []

    base_idx = []
    idx0 = 0
    for _, ring in ring_arrays:
        base_idx.append(idx0)
        points.append(ring)
        idx0 += len(ring)

    points = np.vstack(points)

    for i in range(len(ring_arrays) - 1):
        b0 = base_idx[i]
        b1 = base_idx[i + 1]

        for j in range(n_boundary_pts):
            jn = (j + 1) % n_boundary_pts

            p0 = b0 + j
            p1 = b0 + jn
            p2 = b1 + j
            p3 = b1 + jn

            # two triangles per quad
            faces.append([3, p0, p2, p1])
            faces.append([3, p1, p2, p3])

    faces = np.hstack(faces)
    mesh = pv.PolyData(points, faces)
    return mesh



In [25]:
# =========================================================
# RUN
# =========================================================

# Start with max_edge_length=None.
# If boundary leaks too far outward, try values like 50, 75, 100 depending on your XY units.
syn_gdf, z_levels = build_delaunay_slices(
    gdf=gdf_edit,
    z_step=2.5,
    dz_window=2.5,
    n_boundary_pts=300,
    min_points_per_slice=12,
    max_edge_length=1000
)

print("Synthetic points:", 0 if syn_gdf is None else len(syn_gdf))

if syn_gdf is not None:
    mesh = mesh_between_slices(syn_gdf, n_boundary_pts=180)
else:
    mesh = None


Synthetic points: 59400
Need at least 2 valid slices to mesh.


In [26]:
# =========================================================
# OPTIONAL: COMBINE REAL + SYNTHETIC
# =========================================================

if syn_gdf is not None:
    gdf_real = gdf_edit.copy()
    gdf_real["source"] = "real"

    common_cols = sorted(set(gdf_real.columns).union(set(syn_gdf.columns)))
    for col in common_cols:
        if col not in gdf_real.columns:
            gdf_real[col] = np.nan
        if col not in syn_gdf.columns:
            syn_gdf[col] = np.nan

    gdf_combined = pd.concat(
        [gdf_real[common_cols], syn_gdf[common_cols]],
        ignore_index=True
    )
    gdf_combined = gpd.GeoDataFrame(gdf_combined, geometry="geometry", crs=gdf_edit.crs)


# =========================================================
# PLOT
# =========================================================

plotter = pv.Plotter()

real_coords = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])
plotter.add_mesh(
    pv.PolyData(real_coords),
    color="red",
    point_size=5,
    render_points_as_spheres=True,
    label="Real points"
)

if syn_gdf is not None:
    syn_coords = np.array([(g.x, g.y, g.z) for g in syn_gdf.geometry])
    plotter.add_mesh(
        pv.PolyData(syn_coords),
        color="cyan",
        point_size=3,
        render_points_as_spheres=True,
        label="Synthetic slice points"
    )

if mesh is not None:
    plotter.add_mesh(
        mesh,
        opacity=0.35,
        show_edges=False,
        label="Delaunay lofted mesh"
    )

plotter.add_legend()
plotter.show()


Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26b4a3e5350_6&reconnect=auto" class="pyvis…

Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\ProgramData\anaconda3\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


In [28]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import polygonize, unary_union
from scipy.spatial import Delaunay
from collections import defaultdict, deque
import pyvista as pv


# =========================================================
# BASIC HELPERS
# =========================================================

def extract_xyz_from_gdf(gdf):
    out = gdf.copy()
    out["x"] = out.geometry.x
    out["y"] = out.geometry.y
    out["z"] = out.geometry.apply(lambda g: g.z if g.has_z else np.nan)
    out = out.dropna(subset=["x", "y", "z"]).copy()
    return out


def resample_closed_ring(xy, n_samples=180):
    """
    Resample a closed XY ring to fixed point count.
    """
    xy = np.asarray(xy)

    if len(xy) < 4:
        return None

    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seglen = np.sqrt(np.sum(np.diff(xy, axis=0) ** 2, axis=1))
    s = np.concatenate([[0.0], np.cumsum(seglen)])
    total = s[-1]

    if total <= 0:
        return None

    target = np.linspace(0, total, n_samples + 1)[:-1]
    x_new = np.interp(target, s, xy[:, 0])
    y_new = np.interp(target, s, xy[:, 1])

    return np.column_stack([x_new, y_new])


def polygon_area_signed(xy):
    """
    Signed area for orientation check.
    """
    x = xy[:, 0]
    y = xy[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - y * np.roll(x, -1))


def ensure_ccw(xy):
    """
    Make ring counterclockwise.
    """
    return xy if polygon_area_signed(xy) > 0 else xy[::-1]


def rotate_ring_to_reference(ring_xy, ref_xy):
    """
    Circularly shift ring so it best aligns with reference ring.
    Both must have same number of points.
    """
    n = len(ring_xy)
    best_shift = 0
    best_score = np.inf

    for k in range(n):
        test = np.roll(ring_xy, shift=k, axis=0)
        score = np.mean(np.sum((test - ref_xy) ** 2, axis=1))
        if score < best_score:
            best_score = score
            best_shift = k

    return np.roll(ring_xy, shift=best_shift, axis=0)


def maybe_reverse_ring_to_reference(ring_xy, ref_xy):
    """
    Flip ordering if reversed ring matches reference better.
    """
    forward = np.mean(np.sum((ring_xy - ref_xy) ** 2, axis=1))
    reverse = np.mean(np.sum((ring_xy[::-1] - ref_xy) ** 2, axis=1))
    return ring_xy if forward <= reverse else ring_xy[::-1]


def ring_centroid(xy):
    return np.mean(xy, axis=0)


# =========================================================
# DELAUNAY COMPONENT SPLITTING
# =========================================================

def build_valid_triangle_list(points_xy, simplices, max_edge_length=None):
    """
    Keep only triangles whose maximum edge length <= max_edge_length.
    If max_edge_length is None, keep all.
    """
    valid = []

    for tri_idx, simp in enumerate(simplices):
        p0, p1, p2 = points_xy[simp]

        e01 = np.linalg.norm(p0 - p1)
        e12 = np.linalg.norm(p1 - p2)
        e20 = np.linalg.norm(p2 - p0)

        if max_edge_length is not None:
            if max(e01, e12, e20) > max_edge_length:
                continue

        valid.append(tri_idx)

    return valid


def triangle_components_from_delaunay(points_xy, max_edge_length=None):
    """
    Split Delaunay triangles into connected components.
    Two triangles are connected if they share an edge.
    """
    if len(points_xy) < 3:
        return None, None

    tri = Delaunay(points_xy)
    simplices = tri.simplices

    valid_tri_ids = build_valid_triangle_list(points_xy, simplices, max_edge_length=max_edge_length)

    if len(valid_tri_ids) == 0:
        return simplices, []

    edge_to_triangles = defaultdict(list)

    for tri_idx in valid_tri_ids:
        simp = simplices[tri_idx]
        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0]))),
        ]
        for e in edges:
            edge_to_triangles[e].append(tri_idx)

    adjacency = defaultdict(set)
    for edge, tri_list in edge_to_triangles.items():
        if len(tri_list) > 1:
            for i in tri_list:
                for j in tri_list:
                    if i != j:
                        adjacency[i].add(j)

    visited = set()
    components = []

    for tri_idx in valid_tri_ids:
        if tri_idx in visited:
            continue

        comp = []
        q = deque([tri_idx])
        visited.add(tri_idx)

        while q:
            cur = q.popleft()
            comp.append(cur)
            for nb in adjacency[cur]:
                if nb not in visited:
                    visited.add(nb)
                    q.append(nb)

        components.append(comp)

    return simplices, components


def polygon_from_triangle_component(points_xy, simplices, component_tri_ids):
    """
    Build boundary polygon from one connected triangle component.
    Boundary edges are edges used by only one triangle within the component.
    """
    edge_count = defaultdict(int)

    for tri_idx in component_tri_ids:
        simp = simplices[tri_idx]
        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0]))),
        ]
        for e in edges:
            edge_count[e] += 1

    boundary_edges = [e for e, c in edge_count.items() if c == 1]
    if len(boundary_edges) < 3:
        return None

    lines = [LineString([points_xy[i], points_xy[j]]) for i, j in boundary_edges]
    merged = unary_union(lines)
    polys = list(polygonize(merged))

    if len(polys) == 0:
        return None

    poly = max(polys, key=lambda p: p.area)
    return poly


# =========================================================
# BUILD SEPARATE RINGS PER SLICE
# =========================================================

def build_multi_component_slices(
    gdf,
    z_step=2.5,
    dz_window=7.5,
    n_boundary_pts=180,
    min_points_per_slice=12,
    max_edge_length=100.0,
    min_component_area=1.0,
):
    """
    For each z slice:
      - gather all points in z-window
      - Delaunay triangulate XY
      - split into connected components
      - polygonize each component
      - resample boundary ring
    """
    work = extract_xyz_from_gdf(gdf)

    zmin = work["z"].min()
    zmax = work["z"].max()
    z_levels = np.arange(zmin, zmax + z_step, z_step)

    slice_rings = []

    for z0 in z_levels:
        sl = work[(work["z"] >= z0 - dz_window) & (work["z"] <= z0 + dz_window)].copy()

        if len(sl) < min_points_per_slice:
            continue

        pts = sl[["x", "y"]].to_numpy()

        simplices, components = triangle_components_from_delaunay(
            pts,
            max_edge_length=max_edge_length
        )

        if simplices is None or len(components) == 0:
            continue

        local_comp_id = 0

        for comp in components:
            poly = polygon_from_triangle_component(pts, simplices, comp)
            if poly is None:
                continue

            if poly.area < min_component_area:
                continue

            xy = np.array(poly.exterior.coords)
            ring = resample_closed_ring(xy, n_samples=n_boundary_pts)
            if ring is None:
                continue

            ring = ensure_ccw(ring)

            cx, cy = ring_centroid(ring)

            ring_df = pd.DataFrame({
                "x": ring[:, 0],
                "y": ring[:, 1],
                "z": z0,
                "z_slice": z0,
                "local_component": local_comp_id,
                "component_area": poly.area,
                "slice_n_input_pts": len(sl),
                "centroid_x": cx,
                "centroid_y": cy,
                "source": "synthetic_delaunay_component"
            })

            slice_rings.append(ring_df)
            local_comp_id += 1

    if len(slice_rings) == 0:
        print("No slice rings created.")
        return None

    syn_df = pd.concat(slice_rings, ignore_index=True)
    syn_df["geometry"] = [Point(x, y, z) for x, y, z in zip(syn_df["x"], syn_df["y"], syn_df["z"])]
    syn_gdf = gpd.GeoDataFrame(syn_df, geometry="geometry", crs=gdf.crs)

    return syn_gdf


# =========================================================
# TRACK COMPONENTS THROUGH DEPTH
# =========================================================

def assign_global_component_ids(syn_gdf, n_boundary_pts=180, max_match_distance=150.0):
    """
    Track slice components between consecutive z levels.
    Each ring gets a persistent global component id.

    Matching is based on centroid distance.
    If 2 separate caverns merge into 1, the lower single ring will be assigned
    to whichever upper component is closest. That is okay for meshing; unmatched
    branches simply terminate.
    """
    syn = syn_gdf.copy()

    z_vals = np.sort(syn["z_slice"].unique())

    ring_records = []
    for z0 in z_vals:
        sub = syn[syn["z_slice"] == z0]
        for local_comp in sorted(sub["local_component"].unique()):
            rr = sub[sub["local_component"] == local_comp].copy()
            if len(rr) != n_boundary_pts:
                continue

            cx = rr["centroid_x"].iloc[0]
            cy = rr["centroid_y"].iloc[0]

            ring_records.append({
                "z_slice": z0,
                "local_component": local_comp,
                "centroid_x": cx,
                "centroid_y": cy
            })

    if len(ring_records) == 0:
        print("No ring records found.")
        return syn

    ring_df = pd.DataFrame(ring_records).sort_values(["z_slice", "local_component"]).reset_index(drop=True)
    ring_df["global_component"] = -1

    next_global_id = 0
    prev_rows = None

    for z0 in z_vals:
        cur_mask = ring_df["z_slice"] == z0
        cur_rows = ring_df[cur_mask].copy()

        if prev_rows is None or len(prev_rows) == 0:
            # initialize first slice
            for idx in cur_rows.index:
                ring_df.loc[idx, "global_component"] = next_global_id
                next_global_id += 1
        else:
            used_prev = set()

            for idx in cur_rows.index:
                cx = ring_df.loc[idx, "centroid_x"]
                cy = ring_df.loc[idx, "centroid_y"]

                best_prev_idx = None
                best_dist = np.inf

                for pidx in prev_rows.index:
                    if pidx in used_prev:
                        continue

                    px = ring_df.loc[pidx, "centroid_x"]
                    py = ring_df.loc[pidx, "centroid_y"]
                    d = np.hypot(cx - px, cy - py)

                    if d < best_dist:
                        best_dist = d
                        best_prev_idx = pidx

                if best_prev_idx is not None and best_dist <= max_match_distance:
                    ring_df.loc[idx, "global_component"] = ring_df.loc[best_prev_idx, "global_component"]
                    used_prev.add(best_prev_idx)
                else:
                    ring_df.loc[idx, "global_component"] = next_global_id
                    next_global_id += 1

        prev_rows = ring_df[ring_df["z_slice"] == z0].copy()

    syn = syn.merge(
        ring_df[["z_slice", "local_component", "global_component"]],
        on=["z_slice", "local_component"],
        how="left"
    )

    return syn


# =========================================================
# ALIGN RINGS WITHIN EACH GLOBAL COMPONENT
# =========================================================

def align_component_rings(syn_gdf, n_boundary_pts=180):
    """
    Align rings within each global component to reduce twist.
    """
    syn = syn_gdf.copy()

    aligned_parts = []

    for gc in sorted(syn["global_component"].dropna().unique()):
        comp = syn[syn["global_component"] == gc].copy()
        z_vals = np.sort(comp["z_slice"].unique())

        ref_ring = None

        for z0 in z_vals:
            rr = comp[comp["z_slice"] == z0].copy()
            if len(rr) != n_boundary_pts:
                continue

            xy = rr[["x", "y"]].to_numpy()
            xy = ensure_ccw(xy)

            if ref_ring is not None:
                xy = maybe_reverse_ring_to_reference(xy, ref_ring)
                xy = rotate_ring_to_reference(xy, ref_ring)

            rr.loc[:, "x"] = xy[:, 0]
            rr.loc[:, "y"] = xy[:, 1]
            rr.loc[:, "geometry"] = [Point(x, y, z0) for x, y in xy]

            aligned_parts.append(rr)
            ref_ring = xy.copy()

    if len(aligned_parts) == 0:
        return syn

    out = pd.concat(aligned_parts, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=syn_gdf.crs)

    return out


# =========================================================
# BUILD MESHES SEPARATELY FOR EACH COMPONENT
# =========================================================

def build_component_meshes(syn_gdf, n_boundary_pts=180):
    """
    Build one mesh per global component.
    Only connects adjacent rings within the same component.
    """
    meshes = []

    if "global_component" not in syn_gdf.columns:
        print("global_component column not found.")
        return meshes

    for gc in sorted(syn_gdf["global_component"].dropna().unique()):
        comp = syn_gdf[syn_gdf["global_component"] == gc].copy()
        z_vals = np.sort(comp["z_slice"].unique())

        ring_arrays = []
        for z0 in z_vals:
            rr = comp[comp["z_slice"] == z0].copy()
            if len(rr) != n_boundary_pts:
                continue
            ring_xyz = rr[["x", "y", "z"]].to_numpy()
            ring_arrays.append((z0, ring_xyz))

        if len(ring_arrays) < 2:
            continue

        points = []
        base_idx = []
        idx0 = 0

        for _, ring in ring_arrays:
            base_idx.append(idx0)
            points.append(ring)
            idx0 += len(ring)

        points = np.vstack(points)

        faces = []
        for i in range(len(ring_arrays) - 1):
            b0 = base_idx[i]
            b1 = base_idx[i + 1]

            for j in range(n_boundary_pts):
                jn = (j + 1) % n_boundary_pts

                p0 = b0 + j
                p1 = b0 + jn
                p2 = b1 + j
                p3 = b1 + jn

                faces.append([3, p0, p2, p1])
                faces.append([3, p1, p2, p3])

        faces = np.hstack(faces)
        mesh = pv.PolyData(points, faces)

        mesh.cell_data["global_component"] = np.full(mesh.n_cells, int(gc))
        meshes.append(mesh)

    return meshes


# =========================================================
# RUN
# =========================================================

# Tune this one carefully:
# If bridging still happens, LOWER max_edge_length.
# If slices break too much, RAISE max_edge_length.
MAX_EDGE = 90.0

syn_gdf = build_multi_component_slices(
    gdf=gdf_edit,
    z_step=2.5,
    dz_window=5,
    n_boundary_pts=180,
    min_points_per_slice=12,
    max_edge_length=MAX_EDGE,
    min_component_area=10.0
)

if syn_gdf is None:
    print("No synthetic components built.")
else:
    syn_gdf = assign_global_component_ids(
        syn_gdf,
        n_boundary_pts=180,
        max_match_distance=220.0
    )

    syn_gdf = align_component_rings(
        syn_gdf,
        n_boundary_pts=180
    )

    component_meshes = build_component_meshes(
        syn_gdf,
        n_boundary_pts=180
    )

    print("Synthetic points:", len(syn_gdf))
    print("Number of tracked components:", syn_gdf["global_component"].nunique())
    print("Number of component meshes:", len(component_meshes))


# =========================================================
# OPTIONAL COMBINE WITH REAL
# =========================================================

if syn_gdf is not None:
    gdf_real = gdf_edit.copy()
    gdf_real["source"] = "real"

    common_cols = sorted(set(gdf_real.columns).union(set(syn_gdf.columns)))
    for col in common_cols:
        if col not in gdf_real.columns:
            gdf_real[col] = np.nan
        if col not in syn_gdf.columns:
            syn_gdf[col] = np.nan

    gdf_combined = pd.concat(
        [gdf_real[common_cols], syn_gdf[common_cols]],
        ignore_index=True
    )
    gdf_combined = gpd.GeoDataFrame(gdf_combined, geometry="geometry", crs=gdf_edit.crs)


# =========================================================
# PLOT
# =========================================================

plotter = pv.Plotter()

# real points
real_coords = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])
plotter.add_mesh(
    pv.PolyData(real_coords),
    color="red",
    point_size=5,
    render_points_as_spheres=True,
    label="Real points"
)

# synthetic points
if syn_gdf is not None:
    syn_coords = np.array([(g.x, g.y, g.z) for g in syn_gdf.geometry])
    plotter.add_mesh(
        pv.PolyData(syn_coords),
        color="cyan",
        point_size=3,
        render_points_as_spheres=True,
        label="Synthetic component slices"
    )

# component meshes
if syn_gdf is not None and len(component_meshes) > 0:
    for i, m in enumerate(component_meshes):
        plotter.add_mesh(
            m,
            opacity=0.35,
            show_edges=False,
            label=f"Component mesh {i}"
        )

plotter.add_legend()
plotter.show()


# =========================================================
# OPTIONAL SAVE
# =========================================================

# if syn_gdf is not None:
#     syn_gdf.to_file("synthetic_multi_component_slices.gpkg", driver="GPKG")

# if syn_gdf is not None:
#     gdf_combined.to_file("combined_real_plus_synthetic_multi_component.gpkg", driver="GPKG")

# if syn_gdf is not None and len(component_meshes) > 0:
#     for i, m in enumerate(component_meshes):
#         m.save(f"cavern_component_{i}.vtk")

Synthetic points: 174060
Number of tracked components: 72
Number of component meshes: 55


Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26b85197050_8&reconnect=auto" class="pyvis…

In [29]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import polygonize, unary_union
from scipy.spatial import Delaunay
from sklearn.cluster import DBSCAN
from collections import defaultdict, deque
import pyvista as pv


# =========================================================
# BASIC HELPERS
# =========================================================

def extract_xyz_from_gdf(gdf):
    out = gdf.copy()
    out["x"] = out.geometry.x
    out["y"] = out.geometry.y
    out["z"] = out.geometry.apply(lambda g: g.z if g.has_z else np.nan)
    out = out.dropna(subset=["x", "y", "z"]).copy()
    return out


def resample_closed_ring(xy, n_samples=180):
    xy = np.asarray(xy)

    if len(xy) < 4:
        return None

    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seglen = np.sqrt(np.sum(np.diff(xy, axis=0) ** 2, axis=1))
    s = np.concatenate([[0.0], np.cumsum(seglen)])
    total = s[-1]

    if total <= 0:
        return None

    target = np.linspace(0, total, n_samples + 1)[:-1]
    x_new = np.interp(target, s, xy[:, 0])
    y_new = np.interp(target, s, xy[:, 1])

    return np.column_stack([x_new, y_new])


def polygon_area_signed(xy):
    x = xy[:, 0]
    y = xy[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - y * np.roll(x, -1))


def ensure_ccw(xy):
    return xy if polygon_area_signed(xy) > 0 else xy[::-1]


def rotate_ring_to_reference(ring_xy, ref_xy):
    n = len(ring_xy)
    best_shift = 0
    best_score = np.inf

    for k in range(n):
        test = np.roll(ring_xy, shift=k, axis=0)
        score = np.mean(np.sum((test - ref_xy) ** 2, axis=1))
        if score < best_score:
            best_score = score
            best_shift = k

    return np.roll(ring_xy, shift=best_shift, axis=0)


def maybe_reverse_ring_to_reference(ring_xy, ref_xy):
    forward = np.mean(np.sum((ring_xy - ref_xy) ** 2, axis=1))
    reverse = np.mean(np.sum((ring_xy[::-1] - ref_xy) ** 2, axis=1))
    return ring_xy if forward <= reverse else ring_xy[::-1]


def ring_centroid(xy):
    return np.mean(xy, axis=0)


# =========================================================
# DELAUNAY COMPONENTS INSIDE ONE CLUSTER
# =========================================================

def build_valid_triangle_list(points_xy, simplices, max_edge_length=None):
    valid = []

    for tri_idx, simp in enumerate(simplices):
        p0, p1, p2 = points_xy[simp]

        e01 = np.linalg.norm(p0 - p1)
        e12 = np.linalg.norm(p1 - p2)
        e20 = np.linalg.norm(p2 - p0)

        if max_edge_length is not None and max(e01, e12, e20) > max_edge_length:
            continue

        valid.append(tri_idx)

    return valid


def triangle_components_from_delaunay(points_xy, max_edge_length=None):
    if len(points_xy) < 3:
        return None, []

    tri = Delaunay(points_xy)
    simplices = tri.simplices

    valid_tri_ids = build_valid_triangle_list(points_xy, simplices, max_edge_length=max_edge_length)
    if len(valid_tri_ids) == 0:
        return simplices, []

    edge_to_triangles = defaultdict(list)

    for tri_idx in valid_tri_ids:
        simp = simplices[tri_idx]
        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0]))),
        ]
        for e in edges:
            edge_to_triangles[e].append(tri_idx)

    adjacency = defaultdict(set)
    for edge, tri_list in edge_to_triangles.items():
        if len(tri_list) > 1:
            for i in tri_list:
                for j in tri_list:
                    if i != j:
                        adjacency[i].add(j)

    visited = set()
    components = []

    for tri_idx in valid_tri_ids:
        if tri_idx in visited:
            continue

        comp = []
        q = deque([tri_idx])
        visited.add(tri_idx)

        while q:
            cur = q.popleft()
            comp.append(cur)
            for nb in adjacency[cur]:
                if nb not in visited:
                    visited.add(nb)
                    q.append(nb)

        components.append(comp)

    return simplices, components


def polygon_from_triangle_component(points_xy, simplices, component_tri_ids):
    edge_count = defaultdict(int)

    for tri_idx in component_tri_ids:
        simp = simplices[tri_idx]
        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0]))),
        ]
        for e in edges:
            edge_count[e] += 1

    boundary_edges = [e for e, c in edge_count.items() if c == 1]
    if len(boundary_edges) < 3:
        return None

    lines = [LineString([points_xy[i], points_xy[j]]) for i, j in boundary_edges]
    merged = unary_union(lines)
    polys = list(polygonize(merged))

    if len(polys) == 0:
        return None

    return max(polys, key=lambda p: p.area)


# =========================================================
# BUILD RINGS WITH DBSCAN + DELAUNAY
# =========================================================

def build_multi_component_slices_improved(
    gdf,
    z_step=2.5,
    dz_window=6.0,
    n_boundary_pts=180,
    min_points_per_slice=10,
    min_points_per_cluster=8,
    max_edge_length=110.0,
    min_component_area=2.0,
    dbscan_eps=70.0,
    dbscan_min_samples=5,
):
    """
    Slice by z.
    Within each slice:
      1) cluster XY points with DBSCAN
      2) run Delaunay inside each cluster only
      3) split connected triangle components
      4) extract/resample boundary ring
    """
    work = extract_xyz_from_gdf(gdf)

    zmin = work["z"].min()
    zmax = work["z"].max()
    z_levels = np.arange(zmin, zmax + z_step, z_step)

    slice_rings = []

    for z0 in z_levels:
        sl = work[(work["z"] >= z0 - dz_window) & (work["z"] <= z0 + dz_window)].copy()
        if len(sl) < min_points_per_slice:
            continue

        pts_all = sl[["x", "y"]].to_numpy()

        # Cluster first
        labels = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit_predict(pts_all)
        sl["cluster_id"] = labels

        local_comp_id = 0

        for cid in sorted(sl["cluster_id"].unique()):
            if cid == -1:
                continue

            sub = sl[sl["cluster_id"] == cid].copy()
            if len(sub) < min_points_per_cluster:
                continue

            pts = sub[["x", "y"]].to_numpy()

            try:
                simplices, components = triangle_components_from_delaunay(
                    pts,
                    max_edge_length=max_edge_length
                )
            except Exception:
                continue

            if simplices is None or len(components) == 0:
                continue

            for comp in components:
                poly = polygon_from_triangle_component(pts, simplices, comp)
                if poly is None:
                    continue

                if poly.area < min_component_area:
                    continue

                xy = np.array(poly.exterior.coords)
                ring = resample_closed_ring(xy, n_samples=n_boundary_pts)
                if ring is None:
                    continue

                ring = ensure_ccw(ring)
                cx, cy = ring_centroid(ring)

                ring_df = pd.DataFrame({
                    "x": ring[:, 0],
                    "y": ring[:, 1],
                    "z": z0,
                    "z_slice": z0,
                    "local_component": local_comp_id,
                    "cluster_id": cid,
                    "component_area": poly.area,
                    "slice_n_input_pts": len(sl),
                    "cluster_n_input_pts": len(sub),
                    "centroid_x": cx,
                    "centroid_y": cy,
                    "source": "synthetic_dbscan_delaunay"
                })

                slice_rings.append(ring_df)
                local_comp_id += 1

    if len(slice_rings) == 0:
        print("No slice rings created.")
        return None

    syn_df = pd.concat(slice_rings, ignore_index=True)
    syn_df["geometry"] = [Point(x, y, z) for x, y, z in zip(syn_df["x"], syn_df["y"], syn_df["z"])]
    syn_gdf = gpd.GeoDataFrame(syn_df, geometry="geometry", crs=gdf.crs)
    return syn_gdf


# =========================================================
# IMPROVED GLOBAL TRACKING
# =========================================================

def assign_global_component_ids_improved(
    syn_gdf,
    n_boundary_pts=180,
    max_match_distance=260.0,
    lookback_slices=3,
):
    """
    Match each ring to the closest compatible ring from the last few slices.
    This is more robust than only matching to previous slice.
    """
    syn = syn_gdf.copy()

    ring_records = []
    for (z0, lc), sub in syn.groupby(["z_slice", "local_component"]):
        if len(sub) != n_boundary_pts:
            continue

        ring_records.append({
            "z_slice": z0,
            "local_component": lc,
            "centroid_x": sub["centroid_x"].iloc[0],
            "centroid_y": sub["centroid_y"].iloc[0],
            "component_area": sub["component_area"].iloc[0],
        })

    ring_df = pd.DataFrame(ring_records)
    if len(ring_df) == 0:
        return syn

    ring_df = ring_df.sort_values(["z_slice", "local_component"]).reset_index(drop=True)
    ring_df["global_component"] = -1

    z_vals = np.sort(ring_df["z_slice"].unique())
    next_global_id = 0

    for iz, z0 in enumerate(z_vals):
        cur_idx = ring_df.index[ring_df["z_slice"] == z0].tolist()

        if iz == 0:
            for idx in cur_idx:
                ring_df.loc[idx, "global_component"] = next_global_id
                next_global_id += 1
            continue

        candidate_prev_zs = z_vals[max(0, iz - lookback_slices):iz]

        prev_pool = ring_df[
            (ring_df["z_slice"].isin(candidate_prev_zs)) &
            (ring_df["global_component"] >= 0)
        ].copy()

        used_prev = set()

        for idx in cur_idx:
            cx = ring_df.loc[idx, "centroid_x"]
            cy = ring_df.loc[idx, "centroid_y"]
            area = ring_df.loc[idx, "component_area"]

            best_prev = None
            best_score = np.inf

            for pidx in prev_pool.index:
                if pidx in used_prev:
                    continue

                px = prev_pool.loc[pidx, "centroid_x"]
                py = prev_pool.loc[pidx, "centroid_y"]
                parea = prev_pool.loc[pidx, "component_area"]

                d = np.hypot(cx - px, cy - py)
                if d > max_match_distance:
                    continue

                area_ratio = max(area, parea) / max(min(area, parea), 1e-9)
                score = d + 15.0 * abs(np.log(area_ratio))

                if score < best_score:
                    best_score = score
                    best_prev = pidx

            if best_prev is not None:
                ring_df.loc[idx, "global_component"] = prev_pool.loc[best_prev, "global_component"]
                used_prev.add(best_prev)
            else:
                ring_df.loc[idx, "global_component"] = next_global_id
                next_global_id += 1

    syn = syn.merge(
        ring_df[["z_slice", "local_component", "global_component"]],
        on=["z_slice", "local_component"],
        how="left"
    )

    return syn


# =========================================================
# ALIGN RINGS
# =========================================================

def align_component_rings(syn_gdf, n_boundary_pts=180):
    syn = syn_gdf.copy()
    aligned_parts = []

    for gc in sorted(syn["global_component"].dropna().unique()):
        comp = syn[syn["global_component"] == gc].copy()
        z_vals = np.sort(comp["z_slice"].unique())

        ref_ring = None

        for z0 in z_vals:
            rr = comp[comp["z_slice"] == z0].copy()
            if len(rr) != n_boundary_pts:
                continue

            xy = rr[["x", "y"]].to_numpy()
            xy = ensure_ccw(xy)

            if ref_ring is not None:
                xy = maybe_reverse_ring_to_reference(xy, ref_ring)
                xy = rotate_ring_to_reference(xy, ref_ring)

            rr.loc[:, "x"] = xy[:, 0]
            rr.loc[:, "y"] = xy[:, 1]
            rr.loc[:, "geometry"] = [Point(x, y, z0) for x, y in xy]

            aligned_parts.append(rr)
            ref_ring = xy.copy()

    if len(aligned_parts) == 0:
        return syn

    out = pd.concat(aligned_parts, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=syn_gdf.crs)
    return out


# =========================================================
# FILL MISSING SLICES MORE AGGRESSIVELY
# =========================================================

def fill_missing_component_slices_improved(
    syn_gdf,
    n_boundary_pts=180,
    z_step=2.5,
    max_gap_slices=3,
):
    """
    Fill missing slices if a component disappears briefly.
    Example: existing at 2700 and 2710, missing 2702.5 and 2705 and 2707.5.
    """
    syn = syn_gdf.copy()
    filled_parts = [syn]

    for gc in sorted(syn["global_component"].dropna().unique()):
        comp = syn[syn["global_component"] == gc].copy()
        z_vals = np.sort(comp["z_slice"].unique())

        for i in range(len(z_vals) - 1):
            z0 = z_vals[i]
            z1 = z_vals[i + 1]

            gap_steps = int(round((z1 - z0) / z_step)) - 1
            if gap_steps <= 0 or gap_steps > max_gap_slices:
                continue

            r0 = comp[comp["z_slice"] == z0].copy()
            r1 = comp[comp["z_slice"] == z1].copy()

            if len(r0) != n_boundary_pts or len(r1) != n_boundary_pts:
                continue

            xyz0 = r0[["x", "y", "z"]].to_numpy()
            xyz1 = r1[["x", "y", "z"]].to_numpy()

            missing_zs = np.arange(z0 + z_step, z1, z_step)

            for zm in missing_zs:
                t = (zm - z0) / (z1 - z0)
                xyzm = (1 - t) * xyz0 + t * xyz1

                add = pd.DataFrame({
                    "x": xyzm[:, 0],
                    "y": xyzm[:, 1],
                    "z": xyzm[:, 2],
                    "z_slice": zm,
                    "local_component": -1,
                    "cluster_id": -1,
                    "component_area": np.nan,
                    "slice_n_input_pts": 0,
                    "cluster_n_input_pts": 0,
                    "centroid_x": np.mean(xyzm[:, 0]),
                    "centroid_y": np.mean(xyzm[:, 1]),
                    "source": "interpolated_missing_slice",
                    "global_component": gc
                })
                add["geometry"] = [Point(x, y, z) for x, y, z in xyzm]
                filled_parts.append(gpd.GeoDataFrame(add, geometry="geometry", crs=syn_gdf.crs))

    out = pd.concat(filled_parts, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=syn_gdf.crs)
    out = out.sort_values(["global_component", "z_slice"]).reset_index(drop=True)
    return out


# =========================================================
# BUILD MESHES
# =========================================================

def build_component_meshes(syn_gdf, n_boundary_pts=180):
    meshes = []

    if "global_component" not in syn_gdf.columns:
        print("global_component column not found.")
        return meshes

    for gc in sorted(syn_gdf["global_component"].dropna().unique()):
        comp = syn_gdf[syn_gdf["global_component"] == gc].copy()
        z_vals = np.sort(comp["z_slice"].unique())

        ring_arrays = []
        for z0 in z_vals:
            rr = comp[comp["z_slice"] == z0].copy()
            if len(rr) != n_boundary_pts:
                continue
            ring_xyz = rr[["x", "y", "z"]].to_numpy()
            ring_arrays.append((z0, ring_xyz))

        if len(ring_arrays) < 2:
            continue

        points = []
        base_idx = []
        idx0 = 0

        for _, ring in ring_arrays:
            base_idx.append(idx0)
            points.append(ring)
            idx0 += len(ring)

        points = np.vstack(points)

        faces = []
        for i in range(len(ring_arrays) - 1):
            b0 = base_idx[i]
            b1 = base_idx[i + 1]

            for j in range(n_boundary_pts):
                jn = (j + 1) % n_boundary_pts

                p0 = b0 + j
                p1 = b0 + jn
                p2 = b1 + j
                p3 = b1 + jn

                faces.append([3, p0, p2, p1])
                faces.append([3, p1, p2, p3])

        faces = np.hstack(faces)
        mesh = pv.PolyData(points, faces)
        mesh.cell_data["global_component"] = np.full(mesh.n_cells, int(gc))
        meshes.append(mesh)

    return meshes


# =========================================================
# RUN
# =========================================================

N_BOUNDARY = 180

syn_gdf = build_multi_component_slices_improved(
    gdf=gdf_edit,
    z_step=2.5,
    dz_window=6.0,
    n_boundary_pts=N_BOUNDARY,
    min_points_per_slice=10,
    min_points_per_cluster=8,
    max_edge_length=110.0,
    min_component_area=2.0,
    dbscan_eps=70.0,
    dbscan_min_samples=5,
)

if syn_gdf is None:
    print("No synthetic components built.")
else:
    syn_gdf = assign_global_component_ids_improved(
        syn_gdf,
        n_boundary_pts=N_BOUNDARY,
        max_match_distance=260.0,
        lookback_slices=3,
    )

    syn_gdf = align_component_rings(
        syn_gdf,
        n_boundary_pts=N_BOUNDARY
    )

    syn_gdf = fill_missing_component_slices_improved(
        syn_gdf,
        n_boundary_pts=N_BOUNDARY,
        z_step=2.5,
        max_gap_slices=3,
    )

    component_meshes = build_component_meshes(
        syn_gdf,
        n_boundary_pts=N_BOUNDARY
    )

    print("Synthetic points:", len(syn_gdf))
    print("Tracked components:", syn_gdf["global_component"].nunique())
    print("Meshes:", len(component_meshes))


# =========================================================
# OPTIONAL COMBINE WITH REAL
# =========================================================

if syn_gdf is not None:
    gdf_real = gdf_edit.copy()
    gdf_real["source"] = "real"

    common_cols = sorted(set(gdf_real.columns).union(set(syn_gdf.columns)))
    for col in common_cols:
        if col not in gdf_real.columns:
            gdf_real[col] = np.nan
        if col not in syn_gdf.columns:
            syn_gdf[col] = np.nan

    gdf_combined = pd.concat(
        [gdf_real[common_cols], syn_gdf[common_cols]],
        ignore_index=True
    )
    gdf_combined = gpd.GeoDataFrame(gdf_combined, geometry="geometry", crs=gdf_edit.crs)


# =========================================================
# PLOT
# =========================================================

plotter = pv.Plotter()

real_coords = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])
plotter.add_mesh(
    pv.PolyData(real_coords),
    color="red",
    point_size=5,
    render_points_as_spheres=True,
    label="Real points"
)

if syn_gdf is not None:
    syn_coords = np.array([(g.x, g.y, g.z) for g in syn_gdf.geometry])
    plotter.add_mesh(
        pv.PolyData(syn_coords),
        color="cyan",
        point_size=3,
        render_points_as_spheres=True,
        label="Synthetic points"
    )

if syn_gdf is not None and len(component_meshes) > 0:
    for i, m in enumerate(component_meshes):
        plotter.add_mesh(
            m,
            opacity=0.35,
            show_edges=False,
            label=f"Mesh {i}"
        )

plotter.add_legend()
plotter.show()


# =========================================================
# OPTIONAL SAVE
# =========================================================

# if syn_gdf is not None:
#     syn_gdf.to_file("synthetic_multi_component_improved.gpkg", driver="GPKG")

# if syn_gdf is not None:
#     gdf_combined.to_file("combined_real_plus_synthetic_improved.gpkg", driver="GPKG")

# if syn_gdf is not None and len(component_meshes) > 0:
#     for i, m in enumerate(component_meshes):
#         m.save(f"cavern_component_improved_{i}.vtk")

Synthetic points: 54720
Tracked components: 20
Meshes: 15


Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26bb669bfd0_9&reconnect=auto" class="pyvis…

In [30]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import polygonize, unary_union
from scipy.spatial import Delaunay


# ==========================================
# 1. get x y z from gdf_edit
# ==========================================
gdf = gdf_edit.copy()

gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y
gdf["z"] = gdf.geometry.apply(lambda g: g.z if g.has_z else np.nan)

gdf = gdf.dropna(subset=["x", "y", "z"]).copy()


# ==========================================
# 2. settings
# ==========================================
dz = 2.5                  # half window thickness
z_step = 2.5              # slice frequency
max_edge = 100.0          # remove very long triangles
n_boundary_pts = 100      # number of synthetic points per slice


# ==========================================
# 3. helper: resample closed ring
# ==========================================
def resample_ring(xy, n=100):
    if len(xy) < 4:
        return None

    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])

    seglen = np.sqrt(np.sum(np.diff(xy, axis=0)**2, axis=1))
    s = np.concatenate([[0], np.cumsum(seglen)])

    if s[-1] == 0:
        return None

    target = np.linspace(0, s[-1], n + 1)[:-1]
    x_new = np.interp(target, s, xy[:, 0])
    y_new = np.interp(target, s, xy[:, 1])

    return np.column_stack([x_new, y_new])


# ==========================================
# 4. loop over z slices
# ==========================================
z_levels = np.arange(gdf["z"].min(), gdf["z"].max() + z_step, z_step)

all_syn = []

for z0 in z_levels:
    zmin = z0 - dz
    zmax = z0 + dz

    dfs = gdf[(gdf["z"] >= zmin) & (gdf["z"] <= zmax)].copy()

    if len(dfs) < 6:
        continue

    pts = dfs[["x", "y"]].to_numpy()

    # Delaunay triangulation
    tri = Delaunay(pts)

    edge_count = {}

    for simp in tri.simplices:
        p0, p1, p2 = pts[simp]

        e01 = np.linalg.norm(p0 - p1)
        e12 = np.linalg.norm(p1 - p2)
        e20 = np.linalg.norm(p2 - p0)

        # skip triangles that are too large
        if max(e01, e12, e20) > max_edge:
            continue

        edges = [
            tuple(sorted((simp[0], simp[1]))),
            tuple(sorted((simp[1], simp[2]))),
            tuple(sorted((simp[2], simp[0])))
        ]

        for e in edges:
            edge_count[e] = edge_count.get(e, 0) + 1

    # boundary edges = edges used only once
    boundary_edges = [e for e, c in edge_count.items() if c == 1]

    if len(boundary_edges) < 3:
        continue

    lines = [LineString([pts[i], pts[j]]) for i, j in boundary_edges]
    merged = unary_union(lines)
    polys = list(polygonize(merged))

    if len(polys) == 0:
        continue

    # take largest polygon
    poly = max(polys, key=lambda p: p.area)

    xy = np.array(poly.exterior.coords)
    ring = resample_ring(xy, n=n_boundary_pts)

    if ring is None:
        continue

    syn_df = pd.DataFrame({
        "x": ring[:, 0],
        "y": ring[:, 1],
        "z": z0,
        "z_slice": z0,
        "source": "synthetic"
    })

    all_syn.append(syn_df)


# ==========================================
# 5. make synthetic gdf
# ==========================================
syn_df = pd.concat(all_syn, ignore_index=True)

syn_df["geometry"] = [
    Point(x, y, z) for x, y, z in zip(syn_df["x"], syn_df["y"], syn_df["z"])
]

syn_gdf = gpd.GeoDataFrame(syn_df, geometry="geometry", crs=gdf_edit.crs)

print(syn_gdf.head())
print("synthetic points:", len(syn_gdf))

              x              y            z      z_slice     source  \
0  2.624622e+06  642741.064902 -3086.857264 -3086.857264  synthetic   
1  2.624623e+06  642740.453459 -3086.857264 -3086.857264  synthetic   
2  2.624624e+06  642739.842015 -3086.857264 -3086.857264  synthetic   
3  2.624625e+06  642739.230572 -3086.857264 -3086.857264  synthetic   
4  2.624626e+06  642738.619129 -3086.857264 -3086.857264  synthetic   

                                     geometry  
0  POINT Z (2624622.355 642741.065 -3086.857)  
1  POINT Z (2624623.258 642740.453 -3086.857)  
2  POINT Z (2624624.161 642739.842 -3086.857)  
3  POINT Z (2624625.064 642739.231 -3086.857)  
4  POINT Z (2624625.966 642738.619 -3086.857)  
synthetic points: 25100


In [31]:
import pyvista as pv
import numpy as np

real_coords = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])
syn_coords = np.array([(g.x, g.y, g.z) for g in syn_gdf.geometry])

plotter = pv.Plotter()

plotter.add_mesh(
    pv.PolyData(real_coords),
    color="red",
    point_size=5,
    render_points_as_spheres=True,
    label="real"
)

plotter.add_mesh(
    pv.PolyData(syn_coords),
    color="cyan",
    point_size=3,
    render_points_as_spheres=True,
    label="synthetic"
)

plotter.add_legend()
plotter.show()

Widget(value='<iframe src="http://localhost:64028/index.html?ui=P_0x26bb566f110_10&reconnect=auto" class="pyvi…

In [99]:
# 1) rebuild clean
gdf_edit = gdf_out.copy()

# 2) make sure the mask actually matches rows
mask = (
    (gdf_edit["orient"] == "v") &
    (gdf_edit["fname"].str.contains("315-135", case=False, na=False)) &
    (gdf_edit["Well_No"].str.lower() == "ppg4")
)

print("Matched rows:", mask.sum())
print(gdf_edit.loc[mask, ["fname", "Well_No", "orient"]].drop_duplicates())

Matched rows: 237
                  fname Well_No orient
6311  315-135_PPG4.gpkg    ppg4      v


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\ProgramData\anaconda3\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


In [18]:
mask = (
    (gdf_edit["orient"] == "v") &
    (gdf_edit["fname"].str.contains("315-135", case=False, na=False)) &
    (gdf_edit["Well_No"].str.lower() == "ppg4")
)

p = gdf_edit.loc[mask, "geometry"].iloc[0]

print("Original point:", p)
print("Anchor:", WELLHEADS["PPG4"]["Xwh"], WELLHEADS["PPG4"]["Ywh"])

ptest = rotate_point_xy_around_anchor(
    p,
    angle_deg=-90.0,
    x0=WELLHEADS["PPG4"]["Xwh"],
    y0=WELLHEADS["PPG4"]["Ywh"]
)

print("Rotated point:", ptest)

Original point: POINT Z (2624647.2056932286 642719.4201432286 -2619.928819688612)
Anchor: 2624645.602891 642717.817341
Rotated point: POINT Z (2624647.2056932286 642716.2145387713 -2619.928819688612)


In [ ]:

# =========================================================
# CELL 29 — SYNTHETIC MESH FROM EXISTING POINTS
# Builds a closed 3-D surface mesh by:
#   1. Slicing gdf_edit by Z
#   2. Fitting a Delaunay boundary per slice
#   3. Resampling each ring to N evenly-spaced points
#   4. Aligning adjacent rings (no twist)
#   5. Connecting rings into a triangulated PyVista mesh
# =========================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import polygonize, unary_union
from scipy.spatial import Delaunay
import pyvista as pv

# ----------------------------------------------------------
# PARAMETERS  (tweak as needed)
# ----------------------------------------------------------
Z_STEP        = 2.5    # vertical spacing between output slices
DZ_WINDOW     = 2.5    # half-band thickness for collecting real pts
N_RING_PTS    = 180    # points per ring boundary (must stay constant)
MIN_PTS_SLICE = 8      # skip slices with fewer real points
MAX_EDGE_LEN  = None   # set e.g. 100 to trim spiky Delaunay triangles

# ----------------------------------------------------------
# HELPERS
# ----------------------------------------------------------

def _resample_ring(xy, n):
    xy = np.asarray(xy)
    if len(xy) < 4:
        return None
    if not np.allclose(xy[0], xy[-1]):
        xy = np.vstack([xy, xy[0]])
    seg = np.sqrt(np.sum(np.diff(xy, axis=0) ** 2, axis=1))
    s = np.concatenate([[0.0], np.cumsum(seg)])
    if s[-1] <= 0:
        return None
    t = np.linspace(0, s[-1], n + 1)[:-1]
    return np.column_stack([np.interp(t, s, xy[:, 0]),
                            np.interp(t, s, xy[:, 1])])


def _signed_area(xy):
    x, y = xy[:, 0], xy[:, 1]
    return 0.5 * float(np.sum(x * np.roll(y, -1) - y * np.roll(x, -1)))


def _align_ring(ring, ref):
    """Flip direction + rotate start point to best match ref ring."""
    if _signed_area(ring) * _signed_area(ref) < 0:
        ring = ring[::-1]
    n = len(ring)
    shifts = np.array([
        np.mean(np.sum((np.roll(ring, k, axis=0) - ref) ** 2, axis=1))
        for k in range(n)
    ])
    return np.roll(ring, int(np.argmin(shifts)), axis=0)


def _delaunay_boundary(pts_xy, max_edge):
    if len(pts_xy) < 3:
        return None
    tri = Delaunay(pts_xy)
    edge_count = {}
    for simp in tri.simplices:
        p = pts_xy[simp]
        edges = [tuple(sorted((simp[i], simp[j])))
                 for i, j in [(0, 1), (1, 2), (2, 0)]]
        lengths = [np.linalg.norm(p[i] - p[j])
                   for i, j in [(0, 1), (1, 2), (2, 0)]]
        if max_edge is not None and max(lengths) > max_edge:
            continue
        for e in edges:
            edge_count[e] = edge_count.get(e, 0) + 1
    boundary = [e for e, c in edge_count.items() if c == 1]
    if not boundary:
        return None
    lines = [LineString([pts_xy[i], pts_xy[j]]) for i, j in boundary]
    polys = list(polygonize(unary_union(lines)))
    if not polys:
        return None
    return max(polys, key=lambda p: p.area)


# ----------------------------------------------------------
# STEP 1 — extract XYZ and build synthetic rings
# ----------------------------------------------------------

gdf = gdf_edit.copy()
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y
gdf["z"] = gdf.geometry.apply(lambda g: g.z if g.has_z else np.nan)
gdf = gdf.dropna(subset=["x", "y", "z"])

z_levels = np.arange(gdf["z"].min(), gdf["z"].max() + Z_STEP, Z_STEP)

rings_xyz = []   # list of ndarray[N_RING_PTS, 3]
ref_ring  = None

for z0 in z_levels:
    band = gdf[(gdf["z"] >= z0 - DZ_WINDOW) & (gdf["z"] <= z0 + DZ_WINDOW)]
    if len(band) < MIN_PTS_SLICE:
        continue

    poly = _delaunay_boundary(band[["x", "y"]].to_numpy(), MAX_EDGE_LEN)
    if poly is None or poly.area <= 0:
        continue

    ring2d = _resample_ring(np.array(poly.exterior.coords), N_RING_PTS)
    if ring2d is None:
        continue

    if ref_ring is not None:
        ring2d = _align_ring(ring2d, ref_ring)
    ref_ring = ring2d.copy()

    rings_xyz.append(np.column_stack([ring2d, np.full(N_RING_PTS, z0)]))

print(f"Built {len(rings_xyz)} rings from {len(z_levels)} z-levels")

# ----------------------------------------------------------
# STEP 2 — triangulate between adjacent rings
# ----------------------------------------------------------

if len(rings_xyz) < 2:
    raise RuntimeError(
        "Need at least 2 valid rings. "
        "Try lowering MIN_PTS_SLICE or increasing DZ_WINDOW."
    )

all_pts = np.vstack(rings_xyz)
faces   = []
base    = [i * N_RING_PTS for i in range(len(rings_xyz))]

for i in range(len(rings_xyz) - 1):
    b0, b1 = base[i], base[i + 1]
    for j in range(N_RING_PTS):
        jn = (j + 1) % N_RING_PTS
        faces += [[3, b0 + j,  b1 + j,  b0 + jn],
                  [3, b0 + jn, b1 + j,  b1 + jn]]

mesh = pv.PolyData(all_pts, np.hstack(faces))
mesh = mesh.compute_normals(auto_orient_normals=True)

print(f"Mesh: {mesh.n_points} points, {mesh.n_cells} triangles")

# ----------------------------------------------------------
# STEP 3 — visualise
# ----------------------------------------------------------

real_pts = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])

pl = pv.Plotter()
pl.add_mesh(mesh,
            color="lightblue",
            opacity=0.7,
            smooth_shading=True,
            show_edges=False,
            label="Synthetic mesh")
pl.add_mesh(pv.PolyData(real_pts),
            color="red",
            point_size=4,
            render_points_as_spheres=True,
            label="Real points")
pl.add_legend()
pl.add_axes()
pl.show()


In [19]:
print(gdf_out.head())
print(gdf_out.columns.tolist())
print(gdf_out.crs)
print(len(gdf_out))

             fname                                   path orient  depth  azi0  \
0  0-180_PPG2.gpkg  C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg      v    NaN   0.0   
1  0-180_PPG2.gpkg  C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg      v    NaN   0.0   
2  0-180_PPG2.gpkg  C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg      v    NaN   0.0   
3  0-180_PPG2.gpkg  C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg      v    NaN   0.0   
4  0-180_PPG2.gpkg  C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg      v    NaN   0.0   

    azi1  az_used Well_No                                    geometry  
0  180.0     90.0    ppg2  POINT Z (2624173.466 642992.817 -2430.456)  
1  180.0     90.0    ppg2  POINT Z (2624183.497 642992.817 -2430.486)  
2  180.0     90.0    ppg2  POINT Z (2624182.228 642992.817 -2439.265)  
3  180.0     90.0    ppg2  POINT Z (2624184.403 642992.817 -2448.074)  
4  180.0     90.0    ppg2  POINT Z (2624194.275 642992.817 -2448.778)  
['fname', 'path', 'orient', 'depth', 'azi0', 'azi1', 'az_used', 'Well_No', 'geome

In [20]:
from pathlib import Path

out_dir = Path(r"Z:\Louisiana DNR 1138\3.0 Analysis\data\PPG 2 SN 32069\SN 32069_SonarSurvey_1993")
out_dir.mkdir(parents=True, exist_ok=True)

gdf_debug = gdf_out.copy()
gdf_debug["X"] = gdf_debug.geometry.x
gdf_debug["Y"] = gdf_debug.geometry.y
gdf_debug["Z"] = gdf_debug.geometry.z

cols = [
    "fname", "path", "orient", "Well_No",
    "depth", "azi0", "azi1", "az_used",
    "X", "Y", "Z", "geometry"
]
gdf_debug = gdf_debug[cols]

# GeoPackage
gpkg_path = out_dir / "1993_referenced_pointcloud_debug.gpkg"
gdf_debug.to_file(gpkg_path, driver="GPKG")

# CSV
csv_path = out_dir / "1993_referenced_pointcloud_debug.csv"
gdf_debug.drop(columns="geometry").to_csv(csv_path, index=False)

print("Saved:")
print("  ", gpkg_path)
print("  ", csv_path)

Saved:
   Z:\Louisiana DNR 1138\3.0 Analysis\data\PPG 2 SN 32069\SN 32069_SonarSurvey_1993\1993_referenced_pointcloud_debug.gpkg
   Z:\Louisiana DNR 1138\3.0 Analysis\data\PPG 2 SN 32069\SN 32069_SonarSurvey_1993\1993_referenced_pointcloud_debug.csv


In [21]:
summary = (
    gdf_out.groupby(["fname", "orient", "Well_No"], dropna=False)
    .agg(
        n_points=("geometry", "size"),
        depth=("depth", "first"),
        azi0=("azi0", "first"),
        azi1=("azi1", "first"),
        az_used=("az_used", "first"),
    )
    .reset_index()
)

print(summary)

                 fname orient Well_No  n_points   depth   azi0   azi1  az_used
0      0-180_PPG2.gpkg      v    ppg2       249     NaN    0.0  180.0     90.0
1      0-180_PPG4.gpkg      v    ppg4       264     NaN    0.0  180.0     90.0
2     225-45_PPG2.gpkg      v    ppg2       249     NaN  225.0   45.0    135.0
3     225-45_PPG4.gpkg      v    ppg4       495     NaN  225.0   45.0    135.0
4          2600ft.gpkg      h     all        73  2600.0    NaN    NaN      NaN
5     270-90_PPG2.gpkg      v    ppg2       258     NaN  270.0   90.0    180.0
6   270-90_PPG4_1.gpkg      v    ppg4       328     NaN  270.0   90.0    180.0
7   270-90_PPG4_2.gpkg      v    ppg4       276     NaN  270.0   90.0    180.0
8          2700ft.gpkg      h     all       174  2700.0    NaN    NaN      NaN
9     277-97_PPG2.gpkg      v    ppg2       351     NaN  277.0   97.0     90.0
10         2800ft.gpkg      h     all       220  2800.0    NaN    NaN      NaN
11         2810ft.gpkg      h     all       218  281

In [22]:
summary_path = out_dir / "1993_referenced_pointcloud_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary to:", summary_path)

Saved summary to: Z:\Louisiana DNR 1138\3.0 Analysis\data\PPG 2 SN 32069\SN 32069_SonarSurvey_1993\1993_referenced_pointcloud_summary.csv
